# Tutorial 11 · LP/QP with CVXPY — Part B

**~40 minutes · after the worksheet**

Model an allocation LP and projection QP, solve them, and inspect primal and dual checks.

Work in pairs. Before each code cell, write down the qualitative result you expect. The notebook is designed to run top-to-bottom in a fresh kernel.


## 1 · Solve the LP with SciPy and inspect slacks


In [1]:
import numpy as np
from scipy.optimize import linprog, minimize

c=np.array([3.,2.]); A=np.array([[1.,1.],[1.,0.],[0.,1.]]); b=np.array([4.,2.,3.])
res=linprog(-c,A_ub=A,b_ub=b,bounds=[(0,None)]*2,method="highs")
print(res.x, "maximum", -res.fun, "slacks", b-A@res.x)
print("inequality marginals for minimized -profit",res.ineqlin.marginals)


[2. 2.] maximum 10.0 slacks [0. 0. 1.]
inequality marginals for minimized -profit [-2. -1. -0.]


## 2 · Solve the projection QP


In [2]:
z=np.array([3.,1.4])
objective=lambda x:.5*np.sum((x-z)**2)
constraints=[{"type":"ineq","fun":lambda x:b-A@x}]
qp=minimize(objective,[1.,1.],bounds=[(0,None)]*2,constraints=constraints,method="SLSQP")
print(qp.x, qp.fun, "max violation",np.max(A@qp.x-b))


[2.         1.40000001] 0.5 max violation 0.0


## 3 · The same models in CVXPY when installed


In [3]:
try:
    import cvxpy as cp
    x=cp.Variable(2,nonneg=True); con=A@x<=b
    prob=cp.Problem(cp.Maximize(c@x),[con]); prob.solve()
    print("CVXPY LP",x.value,prob.value,"dual",con.dual_value)
    q=cp.Variable(2); prob2=cp.Problem(cp.Minimize(.5*cp.sum_squares(q-z)),[A@q<=b,q>=0]); prob2.solve()
    print("CVXPY QP",q.value,prob2.value)
except ImportError:
    print("Optional: install cvxpy to run this comparison; SciPy results above remain complete.")


CVXPY LP [2. 2.] 9.999999999515227 dual [2.00000000e+00 1.00000000e+00 3.78357037e-11]
CVXPY QP [2.  1.4] 0.5


## Closing check

Write three sentences: one numerical result you verified, one geometric/probabilistic interpretation, and one failure mode you would now test in a larger implementation.
